In [1]:

import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/classification220718/sample_submission.csv
/kaggle/input/classification220718/data_description.txt
/kaggle/input/classification220718/train.csv
/kaggle/input/classification220718/test.csv


# discrete list(one-hot)

- workclass#모름 항목이있지만 어차피 원핫
- education
- marital-status
- occupation
- relationship
- race
- native-country

# str binary

- sex#그러나 숫자형으로 변환 필요(modify)

# binary list

- capital-gain-exist#캡 게인 여부 새로(new)
- capital-loss-exist#캡 로스 여부 새로(new)
- capital-gain-over -> 8000넘음(new) #이거 추가해서 valid pred가 올라감

# continous list

- age
- fnlwgt
- education-num
- capital-gain ->8000넘는거 따로 분류
- capital-loss ->
- hours-per-week

# target

- income#binary

In [2]:
#data load
data = pd.read_csv('../input/classification220718/train.csv')
target = data.income
data.drop(['income','no'],axis=1, inplace=True)

index = np.random.permutation(len(data))
train_size = int(len(data)*0.8)
ti = index[:train_size]
vi = index[train_size:]
data_train = data.iloc[ti]
data_valid = data.iloc[vi]
target_train = target.iloc[ti]
target_valid = target.iloc[vi]

In [3]:
'''#continuous features에 대한 군집화 해봄
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

model = KMeans(n_clusters=2, init="random", n_init=1, max_iter=1000, random_state=6).fit(data[[
    'age',
    #'fnlwgt',
    #education-num',
    #'capital-gain',
    'capital-loss'
    #'hours-per-week'
]])
c0, c1 = model.cluster_centers_

print(X_low.shape)
plt.plot(data['age'],data['capital-loss'],'b.')
plt.plot(c0[0], c0[1],marker='^',c='r')
plt.plot(c1[0], c1[1],marker='v',c='y')
'''

'#continuous features에 대한 군집화 해봄\nimport matplotlib.pyplot as plt\nfrom sklearn.cluster import KMeans\n\nmodel = KMeans(n_clusters=2, init="random", n_init=1, max_iter=1000, random_state=6).fit(data[[\n    \'age\',\n    #\'fnlwgt\',\n    #education-num\',\n    #\'capital-gain\',\n    \'capital-loss\'\n    #\'hours-per-week\'\n]])\nc0, c1 = model.cluster_centers_\n\nprint(X_low.shape)\nplt.plot(data[\'age\'],data[\'capital-loss\'],\'b.\')\nplt.plot(c0[0], c0[1],marker=\'^\',c=\'r\')\nplt.plot(c1[0], c1[1],marker=\'v\',c=\'y\')\n'

In [4]:
#preprocessing
#one-hot

train_cat={}#트레인 카테고리 기억용 전역변수

def get_onehot(data, col_label, fit):
    src_col = data.loc[:,col_label]
    cat_1 = None
    if fit:
        cat_1= sorted(list(set(src_col))) #카테고리를 이름순으로 나열
        cat_1.append("nan@!")
        train_cat[col_label] = cat_1
    else:
        cat_1 = train_cat[col_label]
    one_1 = np.zeros((len(src_col), len(cat_1))) #새 컬럼 만들음.
    
    for idx in range(len(src_col)):
        t_cat = src_col.loc[src_col.index[idx]]
        if t_cat in cat_1:
            cat_idx = cat_1.index(t_cat)
        else:
            cat_idx = len(cat_1)-1
        one_1[idx][cat_idx] = 1
    
    dst_col_labels = []
    for cat in cat_1:
        dst_col_labels.append(col_label+'_'+cat)
        
    return pd.DataFrame(one_1, columns=dst_col_labels, index=src_col.index)

#merge
def preprocess(data, fit=False):
    #make new col
    cap_gain = data.loc[:,'capital-gain']#series
    data.loc[:,'capital-gain-exist'] = (cap_gain>0).astype(np.int32)
    data.loc[:,'capital-gain-over'] = (cap_gain>8000).astype(np.int32)
    cap_loss = data.loc[:,'capital-loss']
    data.loc[:,'capital-loss-exist']= (cap_loss>0).astype(np.int32)
    

    #string binary

    sex = data.loc[:,'sex']
    data.loc[:,'sex'] = (sex==' Male').astype(np.int32)
    
    return pd.concat([
        get_onehot(data,'workclass', fit),
        get_onehot(data,'education', fit),
        get_onehot(data,'marital-status', fit),
        get_onehot(data,'occupation', fit),
        get_onehot(data,'relationship', fit),
        get_onehot(data,'race', fit),
        get_onehot(data,'native-country', fit),
        data.drop(['workclass','education','marital-status','occupation','relationship','race','native-country'],axis=1)], axis=1)

model = XGBRegressor(n_estimators=600, learning_rate = 0.03)

In [5]:
from xgboost import XGBClassifier
#est=600,learning_rate=0.03=>traing: 0.887, valid: 0.865, 조금 더 과적합시켜도 될듯.
#est=700,learning_rate=0.02=>traing: 0.890, valid: 0.867, 조금 더 과적합시켜도 될듯.
#est=800,learning_rate=0.02=>traing: 0.892, valid: 0.867, 조금 더...
#800, 0.015 => 0.887, 0.866, 학습이 덜되는걸 보니 트리를 늘려야함.
#900, 0.015 => 0.889, 0.865, 과적합 양상을 보이는걸보니 학습율은 0.02가 적당한듯.
#900, 0.02 => 0.894, 0.867
#1000, 0.02 => 0.895, 0.866, 과적합 양상을 보인다 0.02학습율에선 est 800이 딱이다.-
#800, 0.02 => 0.892, 0.867->결국 test에서 과적합 판정이 났다. 처음으로 돌아가자
model = XGBClassifier(n_estimators=600, learning_rate = 0.03)
model.fit(preprocess(data_train, True),target_train)

/opt/conda/lib/python3.7/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value
/opt/conda/lib/python3.7/site-packages/pandas/core/indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)


XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
              importance_type=None, interaction_constraints='',
              learning_rate=0.03, max_bin=256, max_cat_to_onehot=4,
              max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=600,
              n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
              reg_alpha=0, reg_lambda=1, ...)

In [6]:
pred_valid = model.predict(preprocess(data_valid))
pred_train = model.predict(preprocess(data_train))
print("train pred :",np.mean(target_train==pred_train))
print("valid pred :",np.mean(target_valid==pred_valid))

train pred : 0.8928937041460502
valid pred : 0.8636751407609623


In [7]:
model.fit(preprocess(data, True),target)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
              importance_type=None, interaction_constraints='',
              learning_rate=0.03, max_bin=256, max_cat_to_onehot=4,
              max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=600,
              n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
              reg_alpha=0, reg_lambda=1, ...)

In [8]:
np.mean(target==model.predict(preprocess(data, True)))

0.8876300972530284

In [9]:
test_data = pd.read_csv('../input/classification220718/test.csv')
test_no = test_data.no

test_data.drop(['no'],axis=1, inplace=True)

test_pred = model.predict(preprocess(test_data))

output = pd.DataFrame({'no': test_no,
                       'income': test_pred})
output.to_csv('submission.csv', index=False)